In [1]:
#Import Libraries 
import pandas as pd
import numpy as np
import os

# Make sure we're in the right folder
os.chdir(r"C:\Users\16124\OneDrive\Desktop\Project folder 6")
print(os.getcwd())

C:\Users\16124\OneDrive\Desktop\Project folder 6


In [2]:
#Setting paths 
raw_path = "data/raw/"
prepared_path = "data/prepared/"

os.makedirs(prepared_path, exist_ok=True)
print("Paths ready!")


Paths ready!


In [3]:
# Loading raw data
df = pd.read_csv(raw_path + "U.S._Chronic_Disease_Indicators__CDI___2023_Release.csv", low_memory=False)

print("Shape:", df.shape)

Shape: (1048575, 34)


In [4]:
# Filtering out hte 3 indicators 
df['DataValue'] = pd.to_numeric(df['DataValue'], errors='coerce')

df_diabetes = df[df['Topic'] == 'Diabetes'].copy()
df_obesity = df[df['Question'] == 'Obesity among adults aged >= 18 years'].copy()
df_inactivity = df[df['Question'] == 'No leisure-time physical activity among adults aged >= 18 years'].copy()

df_diabetes['indicator'] = 'diabetes'
df_obesity['indicator'] = 'obesity'
df_inactivity['indicator'] = 'inactivity'

df_filtered = pd.concat([df_diabetes, df_obesity, df_inactivity], ignore_index=True)

print("Filtered shape:", df_filtered.shape)
print(df_filtered['indicator'].value_counts())

Filtered shape: (148077, 35)
indicator
diabetes      132943
obesity         7571
inactivity      7563
Name: count, dtype: int64


In [5]:
# keeping only the columns i need 
cols_needed = [
    'YearStart', 'LocationAbbr', 'LocationDesc', 'LocationID',
    'DataValue', 'LowConfidenceLimit', 'HighConfidenceLimit',
    'StratificationCategory1', 'Stratification1', 'GeoLocation', 'indicator'
]

df_clean = df_filtered[cols_needed].copy()

print("Clean shape:", df_clean.shape)
print(df_clean.head())

Clean shape: (148077, 11)
   YearStart LocationAbbr    LocationDesc  LocationID  DataValue  \
0       2018           SC  South Carolina          45     2657.0   
1       2018           VT         Vermont          50      329.0   
2       2014           WI       Wisconsin          55     2645.0   
3       2011           AL         Alabama           1       35.0   
4       2012           AL         Alabama           1       86.0   

   LowConfidenceLimit  HighConfidenceLimit StratificationCategory1  \
0                 NaN                  NaN                  Gender   
1                 NaN                  NaN                  Gender   
2                 NaN                  NaN                  Gender   
3                 NaN                  NaN                  Gender   
4                 NaN                  NaN                 Overall   

  Stratification1                                    GeoLocation indicator  
0            Male  POINT (-81.04537120699968 33.998821303000454)  d

In [6]:
 # removieng GU — Guam PR — Puerto Rico VI — Virgin Islands AS — American Samoa MP — Northern Mariana Islands to keep only state-level data for map analysis
# Remove US territories and keep only 50 states + DC
states_to_remove = ['GU', 'PR', 'VI', 'AS', 'MP', 'US']

df_clean = df_clean[~df_clean['LocationAbbr'].isin(states_to_remove)]

print("After removing territories:", df_clean.shape)
print(df_clean['LocationAbbr'].nunique(), "unique locations")

After removing territories: (142326, 11)
51 unique locations


In [7]:
## Keeping washington dc as its own territory to avoid any data skewness with virgina and maryland
# creating cdi-overall_widecsv ( the main Tableau file) 
df_wide = df_clean[df_clean['StratificationCategory1'] == 'Overall'].pivot_table(
    index=['LocationAbbr', 'LocationDesc', 'LocationID', 'YearStart'],
    columns='indicator',
    values='DataValue',
    aggfunc='mean'
).reset_index()

df_wide.columns.name = None
df_wide = df_wide.rename(columns={
    'diabetes': 'diabetes_prevalence',
    'obesity': 'obesity_prevalence',
    'inactivity': 'inactivity_prevalence'
})

df_wide = df_wide.dropna(subset=['diabetes_prevalence'])

print("Wide shape:", df_wide.shape)
print(df_wide.head())

Wide shape: (625, 7)
  LocationAbbr LocationDesc  LocationID  YearStart  diabetes_prevalence  \
0           AK       Alaska           2       2010            95.847500   
1           AK       Alaska           2       2011            60.428571   
2           AK       Alaska           2       2012            63.111538   
3           AK       Alaska           2       2013            63.096429   
4           AK       Alaska           2       2014            58.580769   

   inactivity_prevalence  obesity_prevalence  
0                    NaN                 NaN  
1                  22.10               27.50  
2                  18.65               25.70  
3                  22.50               28.55  
4                  19.20               29.75  


In [8]:
# create cdi_demographics 
df_demo = df_clean[df_clean['StratificationCategory1'].isin(['Race/Ethnicity', 'Gender'])].copy()

df_demo = df_demo[['YearStart', 'LocationAbbr', 'LocationDesc', 
                    'StratificationCategory1', 'Stratification1',
                    'DataValue', 'indicator']].copy()

df_demo = df_demo.rename(columns={
    'StratificationCategory1': 'StratCategory',
    'Stratification1': 'Stratification',
    'DataValue': 'prevalence'
})

print("Demo shape:", df_demo.shape)
print(df_demo['StratCategory'].value_counts())

Demo shape: (124065, 7)
StratCategory
Race/Ethnicity    89264
Gender            34801
Name: count, dtype: int64


In [9]:
# creating cdi_overall_long.csv
df_long = df_wide.melt(
    id_vars=['LocationAbbr', 'LocationDesc', 'YearStart'],
    value_vars=['diabetes_prevalence', 'obesity_prevalence', 'inactivity_prevalence'],
    var_name='indicator',
    value_name='prevalence'
)

print("Long shape:", df_long.shape)
print(df_long.head())

Long shape: (1875, 5)
  LocationAbbr LocationDesc  YearStart            indicator  prevalence
0           AK       Alaska       2010  diabetes_prevalence   95.847500
1           AK       Alaska       2011  diabetes_prevalence   60.428571
2           AK       Alaska       2012  diabetes_prevalence   63.111538
3           AK       Alaska       2013  diabetes_prevalence   63.096429
4           AK       Alaska       2014  diabetes_prevalence   58.580769


In [10]:
# Saving all 3 files 
df_wide.to_csv(prepared_path + "cdi_overall_wide.csv", index=False)
df_long.to_csv(prepared_path + "cdi_overall_long.csv", index=False)
df_demo.to_csv(prepared_path + "cdi_demographic.csv", index=False)

print("All 3 files saved to data/prepared/!")
print("Notebook 02 complete!")

All 3 files saved to data/prepared/!
Notebook 02 complete!
